# Chapter 3: Homogeneous Coordinates

**Source orientation.** Perspectives on Projective Geometry, Chapter 3, sections 3.1-3.7, printed pages 47-66 / PDF pages 69-88. The source span was read for structure and terminology; the exposition, examples, code, and diagrams here are original.

**Chapter goal.** Build a working model of the real projective plane `RP^2` from nonzero vectors in `R^3`, then use that model to compute affine charts, elements at infinity, joins, meets, dual statements, projective transformations, and small finite projective planes.

**Guiding question.** How do three homogeneous coordinates remove the exceptional cases from two-dimensional geometry?

The central move is to stop treating a point as only a pair `(x, y)` and instead treat it as a ray through the origin in `R^3`. A line is represented by another homogeneous vector, interpreted as a covector or plane normal. Incidence becomes one dot product, joins and meets become one cross product, and projective transformations become invertible `3 x 3` matrices up to scalar.

## Computational Translation Guide

| Geometric idea | Computational representation | What to inspect |
| --- | --- | --- |
| Projective point in `RP^2` | A nonzero vector `p = (x, y, w)`, with `p` and `lambda p` equivalent for `lambda != 0` | Scaling changes the representative, not the point. |
| Affine chart `w = 1` | If `w != 0`, read the finite point as `(x / w, y / w)` | The chart is a view, not the whole projective plane. |
| Element at infinity | A representative with `w = 0` | It records a direction rather than a finite location. |
| Projective line | A nonzero vector `l = (a, b, c)` standing for `a x + b y + c w = 0` | Incidence is the scalar `p dot l`. |
| Join of two points | `p x q`, a line vector orthogonal to both points | The two dot products with the returned line vanish. |
| Meet of two lines | `l x m`, a point vector orthogonal to both lines | Parallel affine lines meet at a vector with `w = 0`. |
| Duality | Swap point and line roles, and swap join with meet | The same dot product and cross product formulas remain valid. |
| Projective transform | An invertible matrix `M`, with points sent to `M p` and lines to `M^{-T} l` | Incidence and collinearity survive. |
| Finite plane over `GF(q)` | Nonzero triples modulo nonzero scalar multiples, arithmetic mod `q` | Counts become `q^2 + q + 1`, with `q + 1` points on each line. |

Library choices follow the course catalog: NumPy for homogeneous vector arithmetic, SymPy for exact identities, Matplotlib for durable annotated diagrams, Plotly for an inspectable projective transform grid, and pandas for finite-plane validation tables.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives on Projective Geometry root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-03-homogeneous-coordinates"
for child in ["figures", "html", "tables", "checks"]:
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

ARTIFACT_ROOT.relative_to(BOOK_ROOT)


In [ ]:
import itertools
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sympy as sp
from IPython.display import display

from utils.artifacts import assert_artifacts, book_relative, display_artifact, image_stats, save_json, save_table

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 160,
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
})

GENERATED_ARTIFACTS = []
DISPLAY_ARTIFACTS = []
CHECKS = {}

def record(path, display=False):
    path = Path(path)
    GENERATED_ARTIFACTS.append(path)
    if display:
        DISPLAY_ARTIFACTS.append(path)
    return path

def fig_path(name):
    return ARTIFACT_ROOT / "figures" / name

def html_path(name):
    return ARTIFACT_ROOT / "html" / name

def hpoint(x, y, w=1.0):
    return np.array([x, y, w], dtype=float)

def normalize_projective(v, tol=1e-12):
    v = np.asarray(v, dtype=float)
    idx = int(np.argmax(np.abs(v)))
    if abs(v[idx]) < tol:
        raise ValueError("zero vector has no projective class")
    return v / v[idx]

def affine_chart(p, coord=2, tol=1e-12):
    p = np.asarray(p, dtype=float)
    if abs(p[coord]) < tol:
        raise ValueError("point is outside this affine chart")
    keep = [i for i in range(3) if i != coord]
    return p[keep] / p[coord]

def affine_point(p):
    return affine_chart(p, 2)

def join(p, q):
    return np.cross(np.asarray(p, dtype=float), np.asarray(q, dtype=float))

def meet(l, m):
    return np.cross(np.asarray(l, dtype=float), np.asarray(m, dtype=float))

def incidence_value(p, l):
    return float(np.dot(np.asarray(p, dtype=float), np.asarray(l, dtype=float)))

def incident(p, l, tol=1e-9):
    return abs(incidence_value(p, l)) < tol

def plot_affine_line(ax, l, xlim, ylim, **kwargs):
    a, b, c = np.asarray(l, dtype=float)
    if abs(b) >= abs(a) and abs(b) > 1e-12:
        xs = np.linspace(xlim[0], xlim[1], 250)
        ys = -(a * xs + c) / b
    elif abs(a) > 1e-12:
        ys = np.linspace(ylim[0], ylim[1], 250)
        xs = -(b * ys + c) / a
    else:
        return None
    mask = (xs >= xlim[0] - 1e-9) & (xs <= xlim[1] + 1e-9) & (ys >= ylim[0] - 1e-9) & (ys <= ylim[1] + 1e-9)
    return ax.plot(xs[mask], ys[mask], **kwargs)

def project_points(M, pts):
    pts = np.asarray(pts, dtype=float)
    if pts.ndim == 1:
        return M @ pts
    return (M @ pts.T).T

def to_affine_array(pts):
    pts = np.asarray(pts, dtype=float)
    return pts[:, :2] / pts[:, [2]]


## Visualization-Planner Pass

The planner pass turns the source span into a direct implementation brief. Each visual has a specific inspection target and a validation hook. This replaces the previous generic route-map scaffold with chapter-specific constructions.


In [ ]:
visual_rows = [
    {"concept": "RP2 from R3", "artifact": "figures/rp2-rays-affine-chart.png", "representation": "3D rays meeting the w=1 affine chart", "library": "Matplotlib 3D, NumPy", "inspection_target": "Finite rays hit the chart; infinity rays lie in w=0 and do not hit it.", "validation": "finite representatives project to w=1; infinity representatives have w=0"},
    {"concept": "Affine chart atlas", "artifact": "figures/affine-chart-atlas.png", "representation": "two affine charts of the same projective classes", "library": "Matplotlib, NumPy", "inspection_target": "A point invisible in the w-chart can be visible in another chart.", "validation": "chart coordinates reconstruct projectively equivalent representatives"},
    {"concept": "Elements at infinity", "artifact": "figures/infinity-limit-directions.png", "representation": "limit of p + alpha r under homogeneous rescaling", "library": "Matplotlib, NumPy", "inspection_target": "The finite location diverges while the scaled homogeneous vector converges.", "validation": "limit residual decreases and final coordinate tends to zero"},
    {"concept": "Joins and meets via cross product", "artifact": "figures/join-meet-cross-product.png", "representation": "ordinary and parallel intersections computed by cross products", "library": "Matplotlib, NumPy, SymPy", "inspection_target": "The same meet formula returns a finite point or a point at infinity.", "validation": "cross products are orthogonal to both inputs"},
    {"concept": "Parallelism after choosing a line at infinity", "artifact": "figures/parallelism-line-at-infinity.png", "representation": "quadrangle construction with a finite line declared to be infinity", "library": "Matplotlib, NumPy", "inspection_target": "Changing the infinity line changes what the word parallel means.", "validation": "constructed lines pass through the center and the selected infinity points"},
    {"concept": "Point-line duality", "artifact": "figures/duality-primal-dual-configurations.png", "representation": "complete quadrangle and coordinate-dual line arrangement", "library": "Matplotlib, NumPy", "inspection_target": "Six joins of four points dualize to six meets of four lines.", "validation": "incidence counts agree after swapping roles"},
    {"concept": "Projective transformations", "artifact": "html/projective-transform-grid.html", "representation": "interactive grid and circle image under a four-corner homography", "library": "Plotly, NumPy", "inspection_target": "Lines remain lines while distances, angles, and circles need not survive.", "validation": "point images match four targets and line action preserves incidence"},
    {"concept": "Finite projective planes", "artifact": "figures/finite-projective-plane-incidence.png", "representation": "incidence matrices for GF(2) and GF(3)", "library": "Matplotlib, NumPy, pandas", "inspection_target": "Every row and column has q+1 hits; any two points determine one line.", "validation": "counts and uniqueness axioms hold modulo q"},
]

storyboard = {
    "chapter goal": "Use homogeneous coordinates to compute RP^2, infinity, joins/meets, duality, projective maps, and finite projective planes.",
    "source span read": "Chapter 3, sections 3.1-3.7, printed pp. 47-66 / PDF pp. 69-88.",
    "concept inventory": [
        "one-dimensional subspaces of K^3 as projective points",
        "two-dimensional subspaces represented by line normals",
        "affine chart w=1 and other chart choices",
        "points and line at infinity",
        "join and meet as cross products",
        "parallel construction relative to a chosen infinity line",
        "point-line duality",
        "projective transformations and inverse-transpose line action",
        "finite projective planes over GF(2) and GF(3)",
    ],
    "library routing table": visual_rows,
    "visual sequence": visual_rows,
    "artifact plan": [row["artifact"] for row in visual_rows] + ["tables/visual-inspection-targets.csv", "tables/finite-plane-incidence-summary.csv", "checks/homogeneous-coordinate-checks.json", "checks/visual-checks.json", "checks/final-sanity.json"],
    "computational checks": [row["validation"] for row in visual_rows],
    "proof-visualization strategy": "Use invariant trackers: dot-product incidence, cross-product orthogonality, inverse-transpose line action, and finite-model axiom checks.",
    "implementation notes": "All artifacts are generated by this canonical notebook under the chapter-03 artifact subtree; no shared utilities or indexes are modified.",
    "risks": "Projective classes are scale classes, so numeric comparisons use incidence residuals or normalized representatives rather than raw coordinate equality.",
    "acceptance criteria": "Notebook executes with nbclient, artifacts exist and are nonblank, final JSON contains incidence/transform/finite-plane checks, and no generic renderer scaffold remains.",
}
storyboard_path = save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")
targets_path = save_table(visual_rows, ARTIFACT_ROOT, "tables", "visual-inspection-targets.csv")
record(storyboard_path)
record(targets_path)
CHECKS["storyboard_items"] = len(visual_rows)
pd.DataFrame(visual_rows)[["concept", "artifact", "validation"]]


## RP2 from R3, Affine Charts, and Infinity

A homogeneous point is not the vector itself; it is the whole nonzero scalar class along a ray. The `w = 1` affine plane is a convenient screen that many rays hit exactly once. Rays contained in `w = 0` never hit that screen, and those are the points at infinity for this chart.

The chart is not the whole projective plane. The atlas artifact compares the usual `w`-chart with an `x`-chart: a point at infinity for the first chart can become a perfectly finite point in the second. The limit artifact shows why a direction vector `(r_x, r_y, 0)` is the endpoint of moving farther and farther along `p + alpha r`.


In [ ]:
finite_points = [hpoint(-1.1, -0.35), hpoint(0.55, 0.85), hpoint(1.2, -0.65)]
infinity_points = [hpoint(1.0, 0.35, 0.0), hpoint(-0.35, 1.0, 0.0)]

fig = plt.figure(figsize=(8.2, 5.8))
ax = fig.add_subplot(111, projection="3d")
xx, yy = np.meshgrid(np.linspace(-1.4, 1.4, 2), np.linspace(-1.2, 1.2, 2))
ax.plot_surface(xx, yy, np.ones_like(xx), alpha=0.18, color="#78a6c8", linewidth=0)
ax.plot_surface(xx, yy, np.zeros_like(xx), alpha=0.10, color="#d95f02", linewidth=0)
ax.text(1.35, 1.2, 1.02, "affine chart w=1", color="#1f5a7a")
ax.text(1.35, 1.2, 0.02, "infinity plane w=0", color="#9a3d00")
for i, p in enumerate(finite_points, start=1):
    ray = np.vstack([np.zeros(3), 1.35 * p])
    ax.plot(ray[:, 0], ray[:, 1], ray[:, 2], color="#2b6cb0", lw=2)
    ax.scatter([p[0]], [p[1]], [p[2]], color="#2b6cb0", s=35)
    ax.text(p[0], p[1], p[2] + 0.06, f"P{i}")
for i, p in enumerate(infinity_points, start=1):
    ray = np.vstack([-1.2 * p, 1.2 * p])
    ax.plot(ray[:, 0], ray[:, 1], ray[:, 2], color="#c2410c", lw=2)
    ax.scatter([p[0]], [p[1]], [p[2]], color="#c2410c", s=35)
    ax.text(p[0], p[1], p[2] + 0.07, f"I{i}")
ax.scatter([0], [0], [0], color="black", s=20)
ax.text(0, 0, -0.08, "origin")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("w")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.3, 1.3)
ax.set_zlim(-0.1, 1.5)
ax.view_init(elev=22, azim=-58)
ax.set_title("Projective points are rays; the affine plane is one chart")
fig.tight_layout()
rays_path = record(fig_path("rp2-rays-affine-chart.png"), display=True)
fig.savefig(rays_path, bbox_inches="tight")
plt.close(fig)

chart_samples = {
    "finite A": hpoint(-0.8, 0.25, 1.0),
    "finite B": hpoint(0.9, -0.55, 1.0),
    "infinity dir (1, 1)": hpoint(1.0, 1.0, 0.0),
    "infinity dir (1, -0.4)": hpoint(1.0, -0.4, 0.0),
}
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
for ax2, (coord, title, labels) in zip(axes, [(2, "U_w chart: divide by w", ("x/w", "y/w")), (0, "U_x chart: divide by x", ("y/x", "w/x"))]):
    ax2.axhline(0, color="#dddddd", lw=1)
    ax2.axvline(0, color="#dddddd", lw=1)
    for name, p in chart_samples.items():
        try:
            xy = affine_chart(p, coord)
        except ValueError:
            continue
        color = "#2b6cb0" if abs(p[2]) > 1e-12 else "#c2410c"
        ax2.scatter([xy[0]], [xy[1]], s=55, color=color)
        ax2.text(xy[0] + 0.035, xy[1] + 0.035, name, fontsize=8)
    ax2.set_title(title)
    ax2.set_xlabel(labels[0])
    ax2.set_ylabel(labels[1])
    ax2.set_aspect("equal", adjustable="box")
    ax2.grid(True, color="#eeeeee")
axes[0].set_xlim(-1.2, 1.2)
axes[0].set_ylim(-0.8, 0.8)
axes[1].set_xlim(-0.8, 1.2)
axes[1].set_ylim(-0.2, 1.3)
fig.suptitle("The same projective classes seen through two affine charts", y=1.02)
fig.tight_layout()
atlas_path = record(fig_path("affine-chart-atlas.png"), display=True)
fig.savefig(atlas_path, bbox_inches="tight")
plt.close(fig)

p0 = np.array([-0.65, 0.28])
r = np.array([1.0, 0.42])
alphas = np.array([0.0, 0.5, 1.0, 2.0, 5.0, 12.0, 30.0])
finite_path = np.array([p0 + alpha * r for alpha in alphas])
positive = alphas[1:]
scaled_h = np.array([hpoint(*(p0 + alpha * r), 1.0) / alpha for alpha in positive])
limit_h = hpoint(r[0], r[1], 0.0)
limit_errors = np.linalg.norm(scaled_h - limit_h, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].plot(finite_path[:, 0], finite_path[:, 1], "o-", color="#2b6cb0")
for alpha, xy in zip(alphas, finite_path):
    axes[0].text(xy[0] + 0.04, xy[1] + 0.04, f"{alpha:g}", fontsize=8)
axes[0].arrow(p0[0], p0[1], r[0], r[1], width=0.015, color="#c2410c", length_includes_head=True)
axes[0].set_title("Affine motion q_alpha = p + alpha r")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].grid(True, color="#eeeeee")
axes[0].set_aspect("equal", adjustable="datalim")
axes[1].plot(positive, scaled_h[:, 0], "o-", label="x/alpha")
axes[1].plot(positive, scaled_h[:, 1], "o-", label="y/alpha")
axes[1].plot(positive, scaled_h[:, 2], "o-", label="w/alpha")
axes[1].axhline(limit_h[0], color="#1f77b4", ls="--", lw=1)
axes[1].axhline(limit_h[1], color="#ff7f0e", ls="--", lw=1)
axes[1].axhline(0, color="#2ca02c", ls="--", lw=1)
axes[1].set_xscale("log")
axes[1].set_title("Rescaled representatives converge to (r_x, r_y, 0)")
axes[1].set_xlabel("alpha")
axes[1].legend(loc="best", fontsize=8)
axes[1].grid(True, color="#eeeeee")
fig.tight_layout()
limit_path = record(fig_path("infinity-limit-directions.png"), display=True)
fig.savefig(limit_path, bbox_inches="tight")
plt.close(fig)

CHECKS["rp2_finite_points_project_to_chart"] = all(np.isclose(p[2], 1.0) for p in finite_points)
CHECKS["rp2_infinity_points_have_zero_w"] = all(abs(p[2]) < 1e-12 for p in infinity_points)
CHECKS["affine_atlas_infinity_visible_in_x_chart"] = all(np.isfinite(affine_chart(p, 0)).all() for p in infinity_points)
CHECKS["infinity_limit_final_w"] = float(abs(scaled_h[-1, 2]))
CHECKS["infinity_limit_error_decreases"] = bool(limit_errors[-1] < limit_errors[0])
CHECKS["infinity_limit_last_error"] = float(limit_errors[-1])
[book_relative(p) for p in [rays_path, atlas_path, limit_path]]


In [ ]:
for artifact in [rays_path, atlas_path, limit_path]:
    display_artifact(artifact, width=760)


## Joins, Meets, and Parallelism

The equation `p dot l = 0` says that point `p` lies on line `l`. If two point representatives `p` and `q` are distinct, then `p x q` is orthogonal to both, so it represents their join line. If two line representatives `l` and `m` are distinct, then `l x m` is orthogonal to both, so it represents their meet point.

The same formula handles the case where an affine intersection moves to infinity. Parallelism then becomes relative to a selected line at infinity: for a point `p`, a line `l`, and a chosen infinity line `l_inf`, the line through `p` parallel to `l` is `p x (l x l_inf)`.


In [ ]:
def line_from_affine_points(a, b):
    return join(hpoint(*a), hpoint(*b))

def meet_of_segments(A, B, C, D):
    l1 = line_from_affine_points(A, B)
    l2 = line_from_affine_points(C, D)
    return l1, l2, meet(l1, l2)

A = (1.0, 1.0)
B = (3.0, 2.0)
C = (3.0, 0.0)
D1 = (4.0, 1.0)
D2 = (5.0, 1.0)
lAB, lCD1, E1 = meet_of_segments(A, B, C, D1)
_, lCD2, E2 = meet_of_segments(A, B, C, D2)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.6))
for ax, D, lCD, E, title, xlim, ylim in [
    (axes[0], D1, lCD1, E1, "finite meet", (-0.2, 7.5), (-0.2, 4.7)),
    (axes[1], D2, lCD2, E2, "parallel affine lines meet at infinity", (0.2, 5.6), (-0.2, 3.0)),
]:
    for name, xy in {"A": A, "B": B, "C": C, "D": D}.items():
        ax.scatter([xy[0]], [xy[1]], color="#111827", s=35)
        ax.text(xy[0] + 0.05, xy[1] + 0.05, name)
    plot_affine_line(ax, lAB, xlim, ylim, color="#2b6cb0", lw=2, label="AB")
    plot_affine_line(ax, lCD, xlim, ylim, color="#c2410c", lw=2, label="CD")
    if abs(E[2]) > 1e-10:
        e_xy = affine_point(E)
        ax.scatter([e_xy[0]], [e_xy[1]], color="#6d28d9", s=48, zorder=4)
        ax.text(e_xy[0] + 0.08, e_xy[1], f"E = {np.round(E, 2)}")
    else:
        direction = E[:2] / np.linalg.norm(E[:2])
        ax.arrow(3.6, 2.25, 0.7 * direction[0], 0.7 * direction[1], color="#6d28d9", width=0.025, length_includes_head=True)
        ax.text(3.55, 2.55, f"E = {np.round(normalize_projective(E), 2)}")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title)
    ax.grid(True, color="#eeeeee")
    ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
join_meet_path = record(fig_path("join-meet-cross-product.png"), display=True)
fig.savefig(join_meet_path, bbox_inches="tight")
plt.close(fig)

p1, p2, p3, q1, q2, q3 = sp.symbols("p1 p2 p3 q1 q2 q3")
p_sym = sp.Matrix([p1, p2, p3])
q_sym = sp.Matrix([q1, q2, q3])
cross_sym = p_sym.cross(q_sym)
CHECKS["symbolic_cross_product_orthogonal_to_first"] = bool(sp.simplify(p_sym.dot(cross_sym)) == 0)
CHECKS["symbolic_cross_product_orthogonal_to_second"] = bool(sp.simplify(q_sym.dot(cross_sym)) == 0)
CHECKS["join_meet_finite_incidence_residual"] = float(max(abs(incidence_value(hpoint(*A), lAB)), abs(incidence_value(hpoint(*B), lAB)), abs(incidence_value(E1, lAB)), abs(incidence_value(E1, lCD1))))
CHECKS["join_meet_parallel_w_abs"] = float(abs(E2[2]))
CHECKS["join_meet_parallel_direction"] = normalize_projective(E2).round(6).tolist()

Aq = hpoint(-0.771, 0.445)
Bq = hpoint(2.803, -0.469)
Cq = hpoint(1.051, 1.812)
Dq = hpoint(-0.061, 1.572)
ABq, BCq, CDq, DAq = join(Aq, Bq), join(Bq, Cq), join(Cq, Dq), join(Dq, Aq)
ACq, BDq = join(Aq, Cq), join(Bq, Dq)
P_inf_1 = meet(ABq, CDq)
P_inf_2 = meet(BCq, DAq)
chosen_l_inf = join(P_inf_1, P_inf_2)
center = meet(ACq, BDq)
parallel_1 = join(center, P_inf_1)
parallel_2 = join(center, P_inf_2)
fig, ax = plt.subplots(figsize=(7.2, 5.2))
xlim = (-3.3, 3.2)
ylim = (-0.8, 2.9)
for line, color, label in [(ABq, "#9ca3af", "opposite sides"), (CDq, "#9ca3af", None), (BCq, "#9ca3af", None), (DAq, "#9ca3af", None)]:
    plot_affine_line(ax, line, xlim, ylim, color=color, lw=1.4, ls="--", label=label)
for line, color, label in [(ACq, "#7c3aed", "diagonals"), (BDq, "#7c3aed", None), (chosen_l_inf, "#c2410c", "chosen infinity line"), (parallel_1, "#2b6cb0", "constructed parallels"), (parallel_2, "#2b6cb0", None)]:
    plot_affine_line(ax, line, xlim, ylim, color=color, lw=2, label=label)
for name, p in {"A": Aq, "B": Bq, "C": Cq, "D": Dq}.items():
    xy = affine_point(p)
    ax.scatter([xy[0]], [xy[1]], color="#111827", s=40, zorder=4)
    ax.text(xy[0] + 0.04, xy[1] + 0.04, name)
for name, p, color in [("P1", P_inf_1, "#c2410c"), ("P2", P_inf_2, "#c2410c"), ("m", center, "#7c3aed")]:
    xy = affine_point(p)
    ax.scatter([xy[0]], [xy[1]], color=color, s=45, zorder=5)
    ax.text(xy[0] + 0.05, xy[1] + 0.05, name)
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_aspect("equal", adjustable="box")
ax.set_title("A finite line can be selected to play the role of infinity")
ax.grid(True, color="#eeeeee")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
parallel_path = record(fig_path("parallelism-line-at-infinity.png"), display=True)
fig.savefig(parallel_path, bbox_inches="tight")
plt.close(fig)
parallel_residuals = [
    incidence_value(center, parallel_1), incidence_value(P_inf_1, parallel_1),
    incidence_value(center, parallel_2), incidence_value(P_inf_2, parallel_2),
    incidence_value(P_inf_1, chosen_l_inf), incidence_value(P_inf_2, chosen_l_inf),
]
CHECKS["parallelism_chosen_infinity_residual"] = float(max(abs(v) for v in parallel_residuals))
CHECKS["parallelism_infinity_line_is_finite_in_standard_chart"] = bool(abs(chosen_l_inf[2]) > 1e-9)
[book_relative(p) for p in [join_meet_path, parallel_path]]


In [ ]:
for artifact in [join_meet_path, parallel_path]:
    display_artifact(artifact, width=760)


## Point-Line Duality and Projective Transformations

The coordinate model makes duality visible. A point vector `(x, y, w)` and a line vector `(a, b, c)` live in algebraically identical projective spaces, and incidence is the symmetric scalar equation `a x + b y + c w = 0`. If a statement uses only point, line, incidence, join, and meet, then swapping point with line and join with meet produces the dual statement.

A projective transformation is represented by an invertible `3 x 3` matrix, with scalar multiples representing the same transformation. It maps point representatives by `p -> M p`. To keep incidence true, a line must transform by `l -> M^{-T} l`, since `(M^{-T} l) dot (M p) = l dot p`. The source chapter's four-point theorem gives a practical way to build such a matrix from four source points and four target points.


In [ ]:
dual_points = {"A": hpoint(-1.15, -0.45), "B": hpoint(1.05, -0.35), "C": hpoint(0.85, 1.1), "D": hpoint(-0.75, 0.95)}
point_items = list(dual_points.items())
join_lines = [(name1 + name2, join(p, q)) for (name1, p), (name2, q) in itertools.combinations(point_items, 2)]
dual_lines = [(name, p.copy()) for name, p in point_items]
dual_meets = [(name1 + name2, meet(l, m)) for (name1, l), (name2, m) in itertools.combinations(dual_lines, 2)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
xlim, ylim = (-1.6, 1.5), (-0.8, 1.4)
for _, line in join_lines:
    plot_affine_line(axes[0], line, xlim, ylim, color="#9ca3af", lw=1.1)
for name, p in point_items:
    xy = affine_point(p)
    axes[0].scatter([xy[0]], [xy[1]], color="#2b6cb0", s=46, zorder=4)
    axes[0].text(xy[0] + 0.04, xy[1] + 0.04, name)
axes[0].set_xlim(*xlim)
axes[0].set_ylim(*ylim)
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_title("Primal: 4 points give 6 joins")
axes[0].grid(True, color="#eeeeee")
xlim, ylim = (-2.7, 2.7), (-2.2, 2.2)
for (name, line), color in zip(dual_lines, ["#2b6cb0", "#c2410c", "#16a34a", "#7c3aed"]):
    plot_affine_line(axes[1], line, xlim, ylim, color=color, lw=1.8, label=f"line {name}")
for label, p in dual_meets:
    if abs(p[2]) > 1e-10:
        xy = affine_point(p)
        if xlim[0] <= xy[0] <= xlim[1] and ylim[0] <= xy[1] <= ylim[1]:
            axes[1].scatter([xy[0]], [xy[1]], color="#111827", s=22, zorder=4)
            axes[1].text(xy[0] + 0.04, xy[1] + 0.04, label, fontsize=7)
axes[1].set_xlim(*xlim)
axes[1].set_ylim(*ylim)
axes[1].set_aspect("equal", adjustable="box")
axes[1].set_title("Dual: same coordinates as 4 lines give 6 meets")
axes[1].grid(True, color="#eeeeee")
axes[1].legend(loc="upper right", fontsize=8)
fig.tight_layout()
duality_path = record(fig_path("duality-primal-dual-configurations.png"), display=True)
fig.savefig(duality_path, bbox_inches="tight")
plt.close(fig)
primal_inc = np.array([[incident(p, l) for _, l in join_lines] for _, p in point_items], dtype=int)
dual_inc = np.array([[incident(p, l) for _, p in dual_meets] for _, l in dual_lines], dtype=int)
CHECKS["duality_primal_join_count"] = len(join_lines)
CHECKS["duality_dual_meet_count"] = len(dual_meets)
CHECKS["duality_each_primal_point_on_three_joins"] = bool((primal_inc.sum(axis=1) == 3).all())
CHECKS["duality_each_dual_line_through_three_meets"] = bool((dual_inc.sum(axis=1) == 3).all())

def frame_from_four(a, b, c, d):
    A = np.column_stack([a, b, c])
    lambdas = np.linalg.solve(A, d)
    return A @ np.diag(lambdas)

def projectivity_from_four(src, dst):
    return frame_from_four(*dst) @ np.linalg.inv(frame_from_four(*src))

def proportional_residual(a, b):
    return float(np.linalg.norm(np.cross(a, b))) / (float(np.linalg.norm(a) * np.linalg.norm(b)) + 1e-15)

src4 = [hpoint(0, 0), hpoint(1, 0), hpoint(1, 1), hpoint(0, 1)]
dst4 = [hpoint(-0.20, 0.05), hpoint(1.55, -0.12), hpoint(1.15, 1.25), hpoint(0.12, 0.82)]
H = projectivity_from_four(src4, dst4)
H = H / np.sign(np.linalg.det(H)) * abs(np.linalg.det(H)) ** (-1 / 3)
corner_residuals = [proportional_residual(H @ s, d) for s, d in zip(src4, dst4)]
fig = make_subplots(rows=1, cols=2, subplot_titles=("source affine grid", "image under projectivity"), horizontal_spacing=0.08)
def add_curve(points, row, col, color, name=None, showlegend=False):
    fig.add_trace(go.Scatter(x=points[:, 0], y=points[:, 1], mode="lines", line=dict(color=color, width=1.5), name=name, showlegend=showlegend), row=row, col=col)
samples = np.linspace(0, 1, 120)
for t in np.linspace(0, 1, 9):
    vertical = np.array([hpoint(t, s) for s in samples])
    horizontal = np.array([hpoint(s, t) for s in samples])
    add_curve(to_affine_array(vertical), 1, 1, "rgba(43,108,176,0.45)")
    add_curve(to_affine_array(horizontal), 1, 1, "rgba(43,108,176,0.45)")
    add_curve(to_affine_array(project_points(H, vertical)), 1, 2, "rgba(194,65,12,0.55)")
    add_curve(to_affine_array(project_points(H, horizontal)), 1, 2, "rgba(194,65,12,0.55)")
for cx, cy, rad in [(0.32, 0.35, 0.16), (0.72, 0.62, 0.13), (0.52, 0.22, 0.10)]:
    theta = np.linspace(0, 2 * np.pi, 180)
    circle = np.array([hpoint(cx + rad * np.cos(t), cy + rad * np.sin(t)) for t in theta])
    add_curve(to_affine_array(circle), 1, 1, "rgba(22,163,74,0.8)")
    add_curve(to_affine_array(project_points(H, circle)), 1, 2, "rgba(22,163,74,0.85)")
src_poly = np.array(src4 + [src4[0]])
dst_poly = project_points(H, src_poly)
add_curve(to_affine_array(src_poly), 1, 1, "black", "four corners", True)
add_curve(to_affine_array(dst_poly), 1, 2, "black", "corner images", True)
fig.update_xaxes(scaleanchor="y", scaleratio=1, row=1, col=1)
fig.update_xaxes(scaleanchor="y2", scaleratio=1, row=1, col=2)
fig.update_layout(width=950, height=460, title="A 3 x 3 projectivity preserves lines but not Euclidean measurements", template="plotly_white", margin=dict(l=30, r=30, t=65, b=30))
transform_path = record(html_path("projective-transform-grid.html"), display=True)
fig.write_html(transform_path, include_plotlyjs=True, full_html=True)
line_before = join(hpoint(0.15, 0.2), hpoint(0.9, 0.72))
point_on_line = hpoint(0.5, 0.2 + (0.72 - 0.2) * (0.5 - 0.15) / (0.9 - 0.15))
line_after = np.linalg.inv(H).T @ line_before
point_after = H @ point_on_line
CHECKS["projectivity_corner_mapping_max_residual"] = float(max(corner_residuals))
CHECKS["projectivity_line_incidence_before"] = float(abs(incidence_value(point_on_line, line_before)))
CHECKS["projectivity_line_incidence_after"] = float(abs(incidence_value(point_after, line_after)))
CHECKS["projectivity_matrix_det_abs"] = float(abs(np.linalg.det(H)))
xs = np.array([-1.4, -0.2, 0.75, 1.6])
def cr(a, b, c, d):
    return ((a - c) * (b - d)) / ((a - d) * (b - c))
ys = (1.1 * xs - 0.25) / (0.22 * xs + 1.0)
CHECKS["cross_ratio_error"] = float(abs(cr(*xs) - cr(*ys)))
[book_relative(p) for p in [duality_path, transform_path]]


In [ ]:
display_artifact(duality_path, width=760)
display_artifact(transform_path, width="100%", height=480)


## Finite Projective Plane Coordinate Experiments

The same homogeneous recipe works over finite fields. For `GF(q)` with prime `q`, normalize nonzero triples modulo scalar multiplication, use the same triples for line coordinates, and declare incidence by `p dot l = 0 mod q`.

The incidence matrices below are not decorative: each black entry is an incidence. The row sums and column sums should all be `q + 1`; every pair of columns should share exactly one row; every pair of rows should share exactly one column.


In [ ]:
def inv_mod(a, q):
    return pow(int(a) % q, -1, q)

def canonical_point_mod(v, q):
    v = tuple(int(x) % q for x in v)
    if v == (0, 0, 0):
        raise ValueError("zero vector")
    if v[2] % q != 0:
        inv = inv_mod(v[2], q)
        return tuple((x * inv) % q for x in v)
    for idx in (0, 1):
        if v[idx] % q != 0:
            inv = inv_mod(v[idx], q)
            return tuple((x * inv) % q for x in v)
    raise ValueError("unreachable")

def canonical_line_mod(v, q):
    v = tuple(int(x) % q for x in v)
    if v == (0, 0, 0):
        raise ValueError("zero vector")
    for idx in range(3):
        if v[idx] % q != 0:
            inv = inv_mod(v[idx], q)
            return tuple((x * inv) % q for x in v)
    raise ValueError("unreachable")

def projective_points_mod(q):
    reps = {canonical_point_mod(v, q) for v in itertools.product(range(q), repeat=3) if v != (0, 0, 0)}
    return sorted(reps, key=lambda p: (p[2] == 0, p[0], p[1], p[2]))

def projective_lines_mod(q):
    reps = {canonical_line_mod(v, q) for v in itertools.product(range(q), repeat=3) if v != (0, 0, 0)}
    return sorted(reps, key=lambda l: (l[0] == 0 and l[1] == 0, l[0], l[1], l[2]))

def incidence_matrix_mod(q):
    points = projective_points_mod(q)
    lines = projective_lines_mod(q)
    mat = np.array([[sum(pi * li for pi, li in zip(p, l)) % q == 0 for p in points] for l in lines], dtype=int)
    return points, lines, mat

def finite_plane_summary(q):
    points, lines, mat = incidence_matrix_mod(q)
    point_pair_counts = [int(np.logical_and(mat[:, i], mat[:, j]).sum()) for i, j in itertools.combinations(range(len(points)), 2)]
    line_pair_counts = [int(np.logical_and(mat[i, :], mat[j, :]).sum()) for i, j in itertools.combinations(range(len(lines)), 2)]
    return {
        "q": q,
        "points": len(points),
        "lines": len(lines),
        "expected_points_lines": q * q + q + 1,
        "line_weight_min": int(mat.sum(axis=1).min()),
        "line_weight_max": int(mat.sum(axis=1).max()),
        "point_weight_min": int(mat.sum(axis=0).min()),
        "point_weight_max": int(mat.sum(axis=0).max()),
        "expected_weight": q + 1,
        "point_pair_unique_line": bool(set(point_pair_counts) == {1}),
        "line_pair_unique_point": bool(set(line_pair_counts) == {1}),
        "finite_points": int(sum(p[2] != 0 for p in points)),
        "infinity_points": int(sum(p[2] == 0 for p in points)),
    }

summaries = [finite_plane_summary(2), finite_plane_summary(3)]
summary_path = save_table(summaries, ARTIFACT_ROOT, "tables", "finite-plane-incidence-summary.csv")
record(summary_path)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8))
for ax, q in zip(axes, [2, 3]):
    points, lines, mat = incidence_matrix_mod(q)
    ax.imshow(mat, cmap="Greys", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"GF({q}): {len(points)} points, {len(lines)} lines")
    ax.set_xlabel("point classes")
    ax.set_ylabel("line classes")
    ax.set_xticks(range(len(points)))
    ax.set_yticks(range(len(lines)))
    ax.set_xticklabels([str(p) for p in points], rotation=90, fontsize=6)
    ax.set_yticklabels([str(l) for l in lines], fontsize=6)
    ax.set_aspect("equal")
    split = q * q - 0.5
    ax.axvline(split, color="#c2410c", lw=1.2)
    ax.text(split + 0.15, -0.65, "infinity pts", color="#c2410c", fontsize=7)
fig.tight_layout()
finite_path = record(fig_path("finite-projective-plane-incidence.png"), display=True)
fig.savefig(finite_path, bbox_inches="tight")
plt.close(fig)
for item in summaries:
    q = item["q"]
    CHECKS[f"gf{q}_point_count"] = item["points"]
    CHECKS[f"gf{q}_line_count"] = item["lines"]
    CHECKS[f"gf{q}_line_weight_ok"] = item["line_weight_min"] == item["line_weight_max"] == item["expected_weight"]
    CHECKS[f"gf{q}_point_weight_ok"] = item["point_weight_min"] == item["point_weight_max"] == item["expected_weight"]
    CHECKS[f"gf{q}_point_pair_unique_line"] = item["point_pair_unique_line"]
    CHECKS[f"gf{q}_line_pair_unique_point"] = item["line_pair_unique_point"]

def parallel_through(point, line, infinity_line):
    return join(point, meet(line, infinity_line))
lab_point = hpoint(0.25, 1.05)
lab_line = join(hpoint(-0.9, -0.15), hpoint(1.15, 0.42))
infinity_choices = {"standard w=0": hpoint(0, 0, 1), "tilted finite line": np.array([0.35, -0.20, 1.0]), "quadrangle infinity line": chosen_l_inf}
lab_rows = []
for name, l_inf in infinity_choices.items():
    infinity_point = meet(lab_line, l_inf)
    result_line = parallel_through(lab_point, lab_line, l_inf)
    lab_rows.append({
        "chosen_infinity_line": name,
        "shared_infinity_point": np.round(normalize_projective(infinity_point), 5).tolist(),
        "result_line": np.round(normalize_projective(result_line), 5).tolist(),
        "passes_through_point_residual": abs(incidence_value(lab_point, result_line)),
        "passes_through_shared_infinity_residual": abs(incidence_value(infinity_point, result_line)),
    })
lab_df = pd.DataFrame(lab_rows)
CHECKS["applied_lab_max_residual"] = float(lab_df[["passes_through_point_residual", "passes_through_shared_infinity_residual"]].to_numpy().max())
display(pd.DataFrame(summaries))
display(lab_df)


In [ ]:
display_artifact(finite_path, width=760)
display(pd.read_csv(summary_path))


## Applied Lab: Change the Infinity Line

The projective formulas do not know which line is the Euclidean line at infinity until you choose one. The lab cell above keeps a finite point and a finite line fixed, then computes the line through the point that is parallel to the fixed line under several choices of `l_inf`. Edit the candidate infinity lines and rerun the cell. The invariant to watch is incidence with the chosen infinity point `l x l_inf`, not Euclidean slope in the original chart.

## Artifact Gallery

These direct links make the notebook readable even outside a live kernel.

![RP2 rays and affine chart](../../artifacts/chapter-03-homogeneous-coordinates/figures/rp2-rays-affine-chart.png)

![Affine chart atlas](../../artifacts/chapter-03-homogeneous-coordinates/figures/affine-chart-atlas.png)

![Infinity as a homogeneous limit](../../artifacts/chapter-03-homogeneous-coordinates/figures/infinity-limit-directions.png)

![Cross-product joins and meets](../../artifacts/chapter-03-homogeneous-coordinates/figures/join-meet-cross-product.png)

![Parallelism from a chosen infinity line](../../artifacts/chapter-03-homogeneous-coordinates/figures/parallelism-line-at-infinity.png)

![Point-line duality](../../artifacts/chapter-03-homogeneous-coordinates/figures/duality-primal-dual-configurations.png)

Open the interactive [projective transform grid](../../artifacts/chapter-03-homogeneous-coordinates/html/projective-transform-grid.html).

![Finite projective plane incidence matrices](../../artifacts/chapter-03-homogeneous-coordinates/figures/finite-projective-plane-incidence.png)

## Takeaways

- `RP^2` is the space of nonzero vectors in `R^3` modulo nonzero scalar multiplication; affine coordinates are chart coordinates, not the objects themselves.
- The line at infinity is not intrinsic. It is a selected projective line, and choosing it determines which affine statements count as parallel.
- Incidence is the dot product `p dot l = 0`; joins and meets are both cross products.
- Point-line duality is already built into the coordinate model: the formulas for incidence, join, and meet are structurally symmetric.
- A projective transformation is an invertible `3 x 3` matrix up to scalar. It sends lines by inverse transpose so incidences survive.
- The finite-field experiment shows that the same coordinate construction gives finite projective planes with `q^2 + q + 1` points and lines when `q` is prime here.


In [ ]:
all_display = [rays_path, atlas_path, limit_path, join_meet_path, parallel_path, duality_path, transform_path, finite_path]
all_required = list(dict.fromkeys([*GENERATED_ARTIFACTS, *all_display]))
assert_artifacts(all_required, min_size=256)

assert CHECKS["rp2_finite_points_project_to_chart"]
assert CHECKS["rp2_infinity_points_have_zero_w"]
assert CHECKS["affine_atlas_infinity_visible_in_x_chart"]
assert CHECKS["infinity_limit_error_decreases"]
assert CHECKS["infinity_limit_final_w"] < 0.04
assert CHECKS["symbolic_cross_product_orthogonal_to_first"]
assert CHECKS["symbolic_cross_product_orthogonal_to_second"]
assert CHECKS["join_meet_finite_incidence_residual"] < 1e-9
assert CHECKS["join_meet_parallel_w_abs"] < 1e-9
assert CHECKS["parallelism_chosen_infinity_residual"] < 1e-9
assert CHECKS["duality_each_primal_point_on_three_joins"]
assert CHECKS["duality_each_dual_line_through_three_meets"]
assert CHECKS["projectivity_corner_mapping_max_residual"] < 1e-9
assert CHECKS["projectivity_line_incidence_before"] < 1e-9
assert CHECKS["projectivity_line_incidence_after"] < 1e-9
assert CHECKS["projectivity_matrix_det_abs"] > 1e-9
assert CHECKS["cross_ratio_error"] < 1e-9
assert CHECKS["applied_lab_max_residual"] < 1e-9
for q in [2, 3]:
    assert CHECKS[f"gf{q}_point_count"] == q * q + q + 1
    assert CHECKS[f"gf{q}_line_count"] == q * q + q + 1
    assert CHECKS[f"gf{q}_line_weight_ok"]
    assert CHECKS[f"gf{q}_point_weight_ok"]
    assert CHECKS[f"gf{q}_point_pair_unique_line"]
    assert CHECKS[f"gf{q}_line_pair_unique_point"]

homogeneous_checks_path = save_json(CHECKS, ARTIFACT_ROOT, "checks", "homogeneous-coordinate-checks.json")
png_stats = {}
for path in [rays_path, atlas_path, limit_path, join_meet_path, parallel_path, duality_path, finite_path]:
    stats = image_stats(path)
    png_stats[book_relative(path)] = {k: v for k, v in stats.items() if k != "path"}
    assert png_stats[book_relative(path)]["pixel_std"] > 1.0

visual_checks = {
    "all_files_exist": all(path.exists() for path in all_required),
    "cross_ratio_error": CHECKS["cross_ratio_error"],
    "numeric_checks": CHECKS,
    "display_artifacts": [book_relative(path) for path in all_display],
    "raster_stats": png_stats,
}
checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")
final = {
    "chapter": 3,
    "source_span": "printed pp. 47-66 / PDF pp. 69-88",
    "notebook_executed": True,
    "artifacts": [book_relative(path) for path in all_required] + [book_relative(homogeneous_checks_path), book_relative(checks_path)],
    "checks": CHECKS,
    "raster_stats": png_stats,
}
final_path = save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
assert_artifacts([homogeneous_checks_path, checks_path, final_path], min_size=256)
{"visual_checks": book_relative(checks_path), "final_sanity": book_relative(final_path), "artifact_count": len(final["artifacts"])}
